In [1]:
import os
import shutil
import re
from functions import *

### Copy txt files

In [18]:
def copy_txt_files(source_dir, dest_dir):
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)
    # Durchlaufen aller Dateien im Quellverzeichnis
    for filename in os.listdir(source_dir):
        print(filename)
        if filename.endswith('.txt'):
            # Vollständigen Pfad der Quelldatei erstellen
            source_file = os.path.join(source_dir, filename)
            # Vollständigen Pfad der Zieldatei erstellen
            dest_file = os.path.join(dest_dir, filename)
            # Datei kopieren
            shutil.copy(source_file, dest_file)
            print(f'Kopiert: {source_file} -> {dest_file}')

In [19]:
source_directory = 'input/ann_text'  # Verzeichnis mit den hochgeladenen Dateien
destination_directory = 'input/half_clinical_trials/'  # Zielverzeichnis

copy_txt_files(source_directory, destination_directory)

### Load n shot Data for Model Prompt

In [10]:
def extract_nct_number(filename):
    match = re.search(r'(NCT\d{8})_(inc|exc)', filename)
    return match.groups() if match else (None, None)

def read_file_content(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        return file.read()

def read_matching_txt_files(study_folder, label_folder, max_files):
    study_filenames = []
    label_filenames = []
    study_contents = []
    label_contents = []

    study_files = [f for f in os.listdir(study_folder) if f.endswith('.txt')][:max_files]
    label_files = [f for f in os.listdir(label_folder) if f.endswith('.txt')][:max_files]
    for filename in study_files:
        nct_number, inc_exc = extract_nct_number(filename)
        if nct_number:
            study_filenames.append(f"{nct_number}_{inc_exc}_study")
            filepath = os.path.join(study_folder, filename)
            study_contents.append(read_file_content(filepath))
    for filename in label_files:
        nct_number, inc_exc = extract_nct_number(filename)
        if nct_number:
            label_filenames.append(f"{nct_number}_{inc_exc}_label")
            filepath = os.path.join(label_folder, filename)
            label_contents.append(read_file_content(filepath))

    return study_filenames, study_contents, label_filenames, label_contents

In [11]:
study_folder = 'input/half_clinical_trials/'  
label_folder = 'chia_label/p2_model_input/' 
max_files = 3  

study_filenames, study_contents, label_filenames, label_contents = read_matching_txt_files(study_folder, label_folder, max_files)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))

study_keys = list(studies.keys())
label_keys = list(labels.keys())
label1 = labels[label_keys[0]]
label2 = labels[label_keys[1]]
label3 = labels[label_keys[2]]

study1 = studies[study_keys[0]]
study2 = studies[study_keys[1]]
study3 = studies[study_keys[2]]

## Model Output to nice JSON and Failure 

In [3]:
model = "Llama-3-70B-Instruct_5_shot"

input_dir = f"model_output/{model}/"
output_dir = f"model_output/{model}/formatted/"
failure_dir = f"model_output/{model}/failed/"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
if not os.path.exists(failure_dir):
    os.makedirs(failure_dir)

for filename in os.listdir(input_dir):
    if filename.endswith('.json'):
        input_file_path = os.path.join(input_dir, filename)
        output_file_path = os.path.join(output_dir, filename)
        failure_file_path = os.path.join(failure_dir, filename)
        try:
            file = read_json(input_file_path)
            print(f"Processing file: {filename}")
            save_json_to_file(file, output_file_path)
        except Exception as e:
            print(f"Error processing file {filename}: {e}")
            shutil.move(input_file_path, failure_file_path)

### Prepare Model Input